# NASDAQ Data Link

[NASDAQ Data Link](https://www.nasdaq.com/nasdaq-data-link) (formerly Quandl) provides access to a wide variety of financial and alternative datasets.

## Getting Started

1. Create an account at [data.nasdaq.com/sign-up](https://data.nasdaq.com/sign-up) - choose **Academic** if you're a student
2. Set up two-factor authentication (Google Authenticator recommended)
3. Get your API key from [data.nasdaq.com/account/profile](https://data.nasdaq.com/account/profile)
4. Store the key as a GitHub Secret (recommended) or in a `.env` file

Install the package:
```
pip install nasdaq-data-link
```

```{figure} ../images/07-nasdaq-home.png
---
name: 07-nasdaq-home.png
align: center
---
NASDAQ Data Link homepage.
```

## Set-up

In [ ]:
import nasdaqdatalink
import os
from dotenv import load_dotenv

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Load API key
load_dotenv()
NASDAQ_API_KEY = os.getenv('NASDAQ_API_KEY')

# Configure the API
nasdaqdatalink.ApiConfig.api_key = NASDAQ_API_KEY

### Secure API Key Usage

For GitHub Codespaces, use secrets:

```python
# Recommended approach
import nasdaqdatalink as ndl
import os

NASDAQ_API_KEY = os.environ.get('NASDAQ')
ndl.ApiConfig.api_key = NASDAQ_API_KEY
```

## Exploring Available Data

Click **EXPLORE** on the main data page to see all available datasets. Much of the data is premium (paid), but you can **filter for free data**.

```{figure} ../images/07-nasdaq-explore.png
---
name: 07-nasdaq-explore.png
align: center
---
Exploring NASDAQ data options.
```

## Example: Zillow Housing Data

NASDAQ provides [Zillow real estate data](https://data.nasdaq.com/databases/ZILLOW) for free.

```{figure} ../images/07-nasdaq-zillow.png
---
name: 07-nasdaq-zillow.png
align: center
---
Zillow housing data on NASDAQ.
```

The data is organized into tables:
- `ZILLOW/DATA` - The actual housing values
- `ZILLOW/REGIONS` - Region identifiers and descriptions
- `ZILLOW/INDICATORS` - Indicator descriptions

### Understanding the Data Structure

Before downloading data, you need to understand what filters are available. Check the [documentation](https://docs.data.nasdaq.com/docs/python-tables) for filtering options.

### Exploring Regions

Let's first pull the regions table to understand what geographic areas are available.

In [ ]:
regions = nasdaqdatalink.get_table('ZILLOW/REGIONS', paginate=True)
regions

```{note}
The `paginate=True` parameter is important! Without it, you only get the first 10,000 rows. With pagination, the limit extends to 1,000,000 rows.
```

In [ ]:
# Filter for just cities
cities = regions[regions.region_type == 'city']
cities

In [ ]:
cities.info()

In [ ]:
# Filter for counties
counties = regions[regions.region_type == 'county']
counties

### Finding Specific Regions

You can search the text in a column to find specific regions. Let's find all NC counties:

In [ ]:
nc_counties = counties[counties['region'].str.contains(';NC')]
nc_counties

In [ ]:
nc_counties.info()

There are 100 counties in NC, so this worked.

### Downloading Housing Data

Now we can use these region IDs to pull actual housing data. First, convert to a list:

In [ ]:
nc_county_list = nc_counties['region_id'].to_list()

Now pull the data for a specific indicator. `ZATT` is one of the housing indicators.

```{warning}
Be careful with large downloads! They can exceed API limits and cause timeouts.
```

In [ ]:
# Example: Pull NC county data (commented to avoid API calls during build)
# zillow_nc = nasdaqdatalink.get_table(
#     'ZILLOW/DATA', 
#     indicator_id='ZATT', 
#     paginate=True, 
#     region_id=nc_county_list,
#     qopts={'columns': ['indicator_id', 'region_id', 'date', 'value']}
# )
# zillow_nc.head(25)

### Exporting Data

You can export to CSV for exploration in Excel:

In [ ]:
# Export counties to CSV
counties.to_csv('counties.csv', index=True)

## Using Rapid API

[Rapid API](https://rapidapi.com/hub) is a marketplace for thousands of APIs - markets, sports, weather, and more. Many have free tiers.

```{figure} ../images/07-rapidapi.png
---
name: 07-rapidapi.png
align: center
---
Rapid API marketplace.
```

### REST APIs

Most APIs on Rapid API are **REST APIs** - a standardized way for computers to communicate using standard data formats like JSON.

Learn more at the [API Learn page](https://rapidapi.com/learn/rest).

### Working with JSON Data

API responses typically come in JSON format. Here's how to handle them:

```python
import requests

# Make API request
response = requests.get(url, headers=headers)

# Convert to JSON
data_json = response.json()

# Flatten nested JSON to DataFrame
df = pd.json_normalize(data=data_json)
```

```{figure} ../images/07-json.png
---
name: 07-json.png
align: center
---
JSON structure - nested dictionaries and lists.
```

For deeply nested JSON, use the `record_path` parameter:

```python
# Flatten nested 'events' key
df = pd.json_normalize(data=data_json, record_path=['events'])
```

## Data on Kaggle

[Kaggle](https://www.kaggle.com) is another great source for datasets. Search their [datasets](https://www.kaggle.com/datasets) for finance-related data.

Examples:
- [Consumer Finance Complaints](https://www.kaggle.com/datasets/kaggle/us-consumer-finance-complaints)
- Stock price histories
- Economic indicators

Kaggle data is usually pre-cleaned for competitions, but you'll still need to do some preparation work.